In [ ]:
import pandas as pd
from pathlib import Path
from predictors.serialization import load_model

In [2]:
test = pd.read_parquet("../data/processed/test.parquet")

In [ ]:
test['multi'] = test['gender'] * 2 + test['age']

## Предсказание моделей на тестовой выборке

In [5]:
models_dir = Path("../models")
models = { }
for file in models_dir.iterdir():
    parts = file.name.split('_')
    
    if parts[0] == 'svc':
        model_name = f"{parts[0]}_{parts[1]}"
        target_idx = 2
    else:
        model_name = parts[0]
        target_idx = 1
    
    target = parts[target_idx]
    
    if model_name not in models:
        models[model_name] = {}
    
    models[model_name][target] = file / "model.pkl"

In [6]:
all_results = pd.DataFrame({ 
    "gender_true": test['gender'],
    "age_true": test['age'],
    "multi_true": test['multi'],
    "length_true": (test['length'] < 2150).astype(int)
}, index=test.index)

In [7]:
models

{'rf': {'age': PosixPath('../models/rf_age_best/model.pkl'),
  'gender': PosixPath('../models/rf_gender_best/model.pkl'),
  'men': PosixPath('../models/rf_men_age_best/model.pkl'),
  'women': PosixPath('../models/rf_women_age_best/model.pkl'),
  'flat': PosixPath('../models/rf_flat_best/model.pkl')},
 'gbm': {'gender': PosixPath('../models/gbm_gender_best/model.pkl'),
  'age': PosixPath('../models/gbm_age_best/model.pkl'),
  'men': PosixPath('../models/gbm_men_age_best/model.pkl'),
  'women': PosixPath('../models/gbm_women_age_best/model.pkl'),
  'flat': PosixPath('../models/gbm_flat_best/model.pkl')},
 'lr': {'gender': PosixPath('../models/lr_gender_best/model.pkl'),
  'age': PosixPath('../models/lr_age_best/model.pkl'),
  'men': PosixPath('../models/lr_men_age_best/model.pkl'),
  'women': PosixPath('../models/lr_women_age_best/model.pkl'),
  'flat': PosixPath('../models/lr_flat_best/model.pkl')},
 'nb': {'gender': PosixPath('../models/nb_gender_best/model.pkl'),
  'age': PosixPath('.

In [8]:
for target in [ 'gender', 'age', 'flat' ]:
    for model in models:
        path = models[model][target]
        pipeline = load_model(path)
        y_pred = pipeline.predict(test)
        y_proba = pipeline.predict_proba(test)
        
        all_results[f'{model}_{target}_pred_label'] = y_pred
        
        for i in pipeline.classes_:
            all_results[f'{model}_{target}_pred_proba_{i}'] = y_proba[:, i]

In [9]:
men = test[test['gender'] == 1]
women = test[test['gender'] == 0]

men_results = pd.DataFrame({ 
    "age_true": men['age'],
}, index=men.index)

women_results = pd.DataFrame({ 
    "age_true": women['age'],
}, index=women.index)

In [10]:
for model in models:
    path = models[model]['men']
    pipeline = load_model(path)
    y_pred = pipeline.predict(men)
    y_proba = pipeline.predict_proba(men)
    
    men_results[f'{model}_pred_label'] = y_pred
    
    for i in pipeline.classes_:
        men_results[f'{model}_pred_proba_{i}'] = y_proba[:, i]

In [11]:
for model in models:
    path = models[model]['women']
    pipeline = load_model(path)
    y_pred = pipeline.predict(women)
    y_proba = pipeline.predict_proba(women)
    
    women_results[f'{model}_pred_label'] = y_pred
    
    for i in pipeline.classes_:
        women_results[f'{model}_pred_proba_{i}'] = y_proba[:, i]

In [12]:
df_combined = pd.concat([women_results, men_results], axis=0)
df_combined.drop(columns=['age_true'], inplace=True)
by_gender_results = df_combined.add_prefix("bygender_")

In [13]:
all = pd.concat([all_results, by_gender_results], axis=1)

In [14]:
experiments_dir = Path('../experiments')
experiments_dir.mkdir(parents=True, exist_ok=True)

In [15]:
all.to_parquet(experiments_dir / "test_pred.parquet")